## Set Price regime

In [35]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

خواندن خروجی مرحله ی قبل

In [36]:
df = pd.read_feather("../Outputs/01_df.feather")

In [37]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [38]:
is_rent = df['cat2_slug'].isin(['commercial-rent', 'residential-rent'])
is_sell = df['cat2_slug'].isin(['commercial-sell', 'residential-sell'])

drop_sale = is_sell & (df["rent_value"].notna() | df["credit_value"].notna() | df["rent_mode"].notna() | df["credit_mode"].notna())
drop_rent = is_rent & (df["price_value"].notna() | df["price_mode"].notna())


df = df.drop(df[drop_sale | drop_rent].index)

In [39]:

def define_price_regime(df):
    df['price_regime'] = 'unknown'
    df['price_status'] = 'unknown'

    # --- Rental rules ---
    is_rent = df['cat2_slug'].isin(['commercial-rent', 'residential-rent'])
    is_sell = df['cat2_slug'].isin(['commercial-sell', 'residential-sell'])

    
    conds_rent = [
        is_rent & (df['rent_mode'] == 'مقطوع') & (df['credit_mode'] == 'مقطوع'),
        is_rent & (df['rent_mode'] == 'مقطوع') & (df['credit_mode'] == 'توافقی') & (df['rent_value'] > 0),
        is_rent & (df['rent_mode'] == 'مقطوع') & (df['credit_mode'] == 'مجانی'),
        is_rent & (df['rent_mode'] == 'توافقی') & (df['credit_mode'] == 'مقطوع'),
        is_rent & (df['rent_mode'] == 'توافقی') & (df['credit_mode'] == 'توافقی'),
        is_rent & (df['rent_mode'] == 'توافقی') & (df['credit_mode'] == 'مجانی'),
        is_rent & (df['rent_mode'] == 'مجانی') & (df['credit_mode'] == 'مقطوع'),
        is_rent & (df['rent_mode'] == 'مجانی') & (df['credit_mode'] == 'توافقی'),
        is_rent & (df['rent_mode'] == 'مجانی') & (df['credit_mode'] == 'مجانی'),
        is_rent & (df['rent_mode'].isna()) & (df['credit_mode'].isna()),
    ]
    
    # English replacements:
    regimes_rent = [
        'mortgage_and_rent',  # رهن و اجاره
        'mortgage_and_rent',
        'rent_only',          # فقط اجاره
        'mortgage_and_rent',
        'mortgage_and_rent',
        'rent_only',
        'mortgage_only',      # فقط رهن
        'mortgage_only',
        '-',
        "NoInformation"
    ]
    
    statuses_rent = [
        'valid',              # معتبر
        'negotiable',         # توافقی
        'valid',
        'negotiable',
        'negotiable',
        'negotiable',
        'valid',
        'negotiable',
        '-',
        'missed'
    ]
    
    rent_regime_arr = np.select(conds_rent, regimes_rent, default='unknown')
    rent_status_arr = np.select(conds_rent, statuses_rent, default='unknown')
    
    df.loc[is_rent, 'price_regime'] = rent_regime_arr[is_rent]
    df.loc[is_rent, 'price_status'] = rent_status_arr[is_rent]

    # --- Sales rules ---
    
    sell_valid = is_sell & (df['price_value'] > 0) & (df['price_mode'] == 'مقطوع')
    sell_nego  = is_sell & (df['price_mode'] == 'توافقی')
    sell_free  = is_sell & (df['price_mode'] == 'مجانی')
    sell_noInfo  = is_sell & (df['price_mode'].isna())
    
    df.loc[sell_valid, 'price_regime'] = 'sell'          # فروش
    df.loc[sell_valid, 'price_status'] = 'valid'         # معتبر
    df.loc[sell_nego, 'price_regime']  = 'sell'
    df.loc[sell_nego, 'price_status']  = 'negotiable'   # توافقی
    df.loc[sell_free, 'price_regime']  = 'sell'
    df.loc[sell_free, 'price_status']  = 'inconsistent' # ناسازگار
    df.loc[sell_noInfo, 'price_regime']  = 'sell'
    df.loc[sell_noInfo, 'price_status']  = 'missed' # مفقود
    
    return df

In [45]:
df = define_price_regime(df)


# جدول ترکیبی
print(df.groupby(['cat2_slug','price_regime', 'price_status']).size())

C:\Users\lenovo\AppData\Local\Temp\ipykernel_17528\668795343.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(['cat2_slug','price_regime', 'price_status']).size())


cat2_slug             price_regime       price_status
commercial-rent       NoInformation      inconsistent         0
                                         missed              82
                                         negotiable           0
                                         unknown              0
                                         valid                0
                      mortgage_and_rent  inconsistent         0
                                         missed               0
                                         negotiable         523
                                         unknown              0
                                         valid            70178
                      mortgage_only      inconsistent         0
                                         missed               0
                                         negotiable           8
                                         unknown              0
                                         valid    

In [41]:
invalid_sell= df[(df["price_regime"]=="unknown") & (df["cat2_slug"]=="commercial-sell")]

invalid_sell[
    [
        "cat2_slug",
        "title",
        "price_mode",
        "price_value",
        "rent_mode",
        "credit_mode",
        "rent_value",
        "credit_value",
        "description"
    ]
].head(20)

,cat2_slug,title,price_mode,price_value,rent_mode,credit_mode,rent_value,credit_value,description


In [42]:
# df = df.drop(df[df["price_regime"] == "Invalid"].index)


In [43]:
df.to_feather("../Outputs/02_df.feather")


In [44]:
# df.to_csv("../Outputs/02_df.csv", index=False, encoding="utf-8-sig")